In [1]:
%load_ext autoreload
%autoreload 2

# general
from pathlib import Path
import re
import numpy as np
import datetime
import matplotlib.pyplot as plt

# spatial
import xarray as xa

# custom
import cbsyst as cb
from cmipper import functions_creche, utils, config, parallelised_download_and_process, main, file_ops

/maps/rt582/miniforge3/envs/shiftpy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Integrating `esgpull`

`esgpull` provides streamlined, maintained functionality to search and download files from the `esgf-node` servers. The job of this notebook is to substitute my own downloading schema (developed as `cmipper`) for `esgpull` while retaining the additional functionality I require. This is:
- ~~Consistent, human-readable file labelling~~ -> Happy with what's given, keeping for simplicity
- Regridding from variable grid to fixed grid
- POTENTIALLY – concatenating and cropping variables

This should enable faster, more reliable, and better-logged downloading of all necessary GCM variables in a 'one file' approach. The user should be able to specify which variables (at what resolution, over a specified time period), which model ensembles, and how many model runs they want for each. The program should then attempt to download these N times.

### Variables
| variable_long_name                                           | variable_id | variable_units | frequency                   |
|--------------------------------------------------------------|-------------|----------------|-----------------------------|
|                                                              |             |                |                             |
| Sea Surface Temperature                                      | tos         | [degC]         | 3hr, Amon, Oday, Odec, **Omon** |
| Sea Water Potential Temperature                              | thetao      | [degC]         | Omon                        |
| Downwelling Shortwave Radiation in Sea Water                 | rsdo        | [W m-2]        | Omon                        |
| Sea Water Salinity                                           | so          | [0.001]        | Omon                        |
| Sea Water X Velocity                                         | uo          | [m s-1]        | Odec, **Omon**                  |
| Sea Water Y Velocity                                         | vo          | [m s-1]        | Odec, **Omon**                  |
| Aragonite Concentration                                      | arag        | [mol m-3]      | Oyr                         |
| Dissolved Nitrate Concentration                              | no3         | [mol m-3]      | **Omon**, Oyr                   |
| Total Dissolved Inorganic Phosphorus Concentration           | po4         | [mol m-3]      | **Omon**, Oyr                   |

Working list:
1. ~~For an example ensemble (`HadGEM3-GC31-MM`) and variable (`tos`), search and download via command-line~~
2. ~~Automate this with a bash script~~ Done to some extent by generating query from yaml file
3. Pipe downloads into regridder

Example query:
`$ esgpull search project:CMIP6 experiment_id:historical institution_id:HadGEM3-GC31-MM variable_id:tos table_id:Omon member_id:r1i1p1f1 --distrib true --show`

In [2]:
# creating searchs from yaml

import itertools
import yaml
import subprocess

# Load YAML config file
with open('/maps/rt582/cmipper/tmp/download_info.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Start the command as a list
query = ["esgpull", "search"]

# Add parameters, handling spaces by keeping each argument separate
for param, value in config.items():
    if not isinstance(value, list):
        # If there's a space, wrap value in quotes
        query.append(f"{param}:{value}" if ' ' not in str(value) else f"{param}:'{value}'")
    else:
        # Join list values with commas and wrap if they have spaces
        joined_values = ','.join([f"'{v}'" if ' ' in str(v) else v for v in value])
        query.append(f"{param}:{joined_values}")

# Add the distribution flag
query.append("--distrib")
query.append("true")

# Display the final command
print("Running:\n", ' '.join(query))

# Uncomment to execute the command
# subprocess.run(query)

Running:
 esgpull search project:CMIP6 nominal_resolution:'25 km' realm:atmos,ocean source_id:ECMWF-IFS-HR institution_id:ECMWF experiment_id:hist-1950,ssp126,ssp585 table_id:Omon,Amon variable_id:tos,rsds,so --distrib true


# Testing processing functions

In [11]:
# write script/function to process (regrid) files at it becomes available

# open test
import xarray as xa
from cmipper import processing

fp = "/maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r1i1p1f1/Omon/vo/gn/v20170915/vo_Omon_ECMWF-IFS-HR_hist-1950_r1i1p1f1_gn_195001-195012.nc"
out = xa.open_dataset(fp)
out

<xarray.Dataset> Size: 5GB
Dimensions:             (time: 12, bnds: 2, lev: 75, i: 1021, j: 1442,
                         vertices: 4)
Coordinates:
  * time                (time) datetime64[ns] 96B 1950-01-16T12:00:00 ... 195...
  * lev                 (lev) float64 600B 0.5058 1.556 ... 5.698e+03 5.902e+03
  * i                   (i) int32 4kB 1 2 3 4 5 6 ... 1017 1018 1019 1020 1021
  * j                   (j) int32 6kB 1 2 3 4 5 6 ... 1438 1439 1440 1441 1442
    latitude            (i, j) float32 6MB ...
    longitude           (i, j) float32 6MB ...
Dimensions without coordinates: bnds, vertices
Data variables:
    time_bnds           (time, bnds) datetime64[ns] 192B ...
    lev_bnds            (lev, bnds) float64 1kB ...
    vertices_latitude   (i, j, vertices) float32 24MB ...
    vertices_longitude  (i, j, vertices) float32 24MB ...
    vo                  (time, lev, i, j) float32 5GB ...
Attributes: (12/51)
    Conventions:             CF-1.7 CMIP-6.0
    activity_id:             HighResMIP
    branch_method:           Initialized directly from parent restart files
    contact:                 chris.roberts@ecmwf.int
    creation_date:           2017-08-11T08:31:39Z
    end_year:                2014
    ...                      ...
    further_info_url:        https://furtherinfo.es-doc.org/CMIP6.ECMWF.ECMWF...
    data_specs_version:      01.00.23
    institution:             European Centre for Medium-Range Weather Forecas...
    references:              Roberts, C. D., Senan, R., Molteni, F., Boussett...
    source:                  ECMWF-IFS-HR (2017): \naerosol: none\natmos: IFS...
    history:                 2017-08-11T08:31:39Z CMOR rewrote data to be con...

In [ ]:
dir_fp = Path("/maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r5i1p1f1/Omon/so/gn/v20190417")
seafloor_inds = processing.find_seafloor_indices_for_directory(dir_fp)


determining seafloor indices... 


# Processing resulting files

In [ ]:
import asyncio
import nest_asyncio
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
from pathlib import Path

# Allow nested event loops for Jupyter notebooks
nest_asyncio.apply()

# Set your top-level raw data directory
raw_data_dir = Path('/maps/rt582/cmipper/.esgpull/data/')
processed_data_dir = Path('/maps/rt582/cmipper/data/test/')

# Function to process files as they appear
async def process_file(file_path):
    # Step 1: Overwrite original (e.g., select pressure level)
    print(f"Processing {file_path} for pressure level selection...")
    # (Your code for overwriting the original file here)
    
    # Step 2: Regrid and save in processed_data_dir
    rel_path = file_path.relative_to(raw_data_dir)
    output_path = processed_data_dir / rel_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Regridding and saving to {output_path}...")
    # (Your code for regridding here, saving to `output_path`)

class DownloadHandler(FileSystemEventHandler):
    # Triggered when a new file is created in raw_data_dir
    def on_created(self, event):
        if not event.is_directory and event.src_path.endswith('.nc'):
            asyncio.run(process_file(Path(event.src_path)))

# Set up observer
event_handler = DownloadHandler()
observer = Observer()
observer.schedule(event_handler, path=raw_data_dir, recursive=True)  # Monitor subdirectories
observer.start()

try:
    while True:
        asyncio.sleep(1)  # Keep the observer running
except KeyboardInterrupt:
    observer.stop()
observer.join()


/tmp/ipykernel_3444880/1855626463.py:41: RuntimeWarning: coroutine 'sleep' was never awaited
  asyncio.sleep(1)  # Keep the observer running


Processing /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r5i1p1f1/Amon/rsds/gr/v20190417/rsds_Amon_ECMWF-IFS-HR_hist-1950_r5i1p1f1_gr_197401-197412.nc for pressure level selection...
Regridding and saving to /maps/rt582/cmipper/data/test/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r5i1p1f1/Amon/rsds/gr/v20190417/rsds_Amon_ECMWF-IFS-HR_hist-1950_r5i1p1f1_gr_197401-197412.nc...
Processing /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r2i1p1f1/Omon/tos/gn/v20181119/tos_Omon_ECMWF-IFS-HR_hist-1950_r2i1p1f1_gn_200901-200912.nc for pressure level selection...
Regridding and saving to /maps/rt582/cmipper/data/test/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r2i1p1f1/Omon/tos/gn/v20181119/tos_Omon_ECMWF-IFS-HR_hist-1950_r2i1p1f1_gn_200901-200912.nc...
Processing /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r4i1p1f1/Omon/so/gn/v20181119/so_Omon_ECMWF-IFS-HR_hist-1950_r4i1p1f1_gn_200601-200

Exception ignored in: <coroutine object sleep at 0x769a2e1d7c40>
Traceback (most recent call last):
  File "/maps/rt582/miniforge3/envs/shiftpy/lib/python3.12/warnings.py", line 553, in _warn_unawaited_coroutine
    warn(msg, category=RuntimeWarning, stacklevel=2, source=coro)
KeyboardInterrupt: 


Processing /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r2i1p1f1/Amon/rsds/gr/v20181119/rsds_Amon_ECMWF-IFS-HR_hist-1950_r2i1p1f1_gr_199101-199112.nc for pressure level selection...
Regridding and saving to /maps/rt582/cmipper/data/test/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r2i1p1f1/Amon/rsds/gr/v20181119/rsds_Amon_ECMWF-IFS-HR_hist-1950_r2i1p1f1_gr_199101-199112.nc...
Processing /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r6i1p1f1/Amon/rsds/gr/v20190417/rsds_Amon_ECMWF-IFS-HR_hist-1950_r6i1p1f1_gr_201301-201312.nc for pressure level selection...
Regridding and saving to /maps/rt582/cmipper/data/test/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r6i1p1f1/Amon/rsds/gr/v20190417/rsds_Amon_ECMWF-IFS-HR_hist-1950_r6i1p1f1_gr_201301-201312.nc...
Processing /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r5i1p1f1/Omon/tos/gn/v20190417/tos_Omon_ECMWF-IFS-HR_hist-1950_r5i1p1f1_gn_1985